# GPU Matrix Multiplication Optimization Tutorial

This notebook guides you through optimizing a matrix multiplication kernel from naive to **50+ TFLOPS**, progressively applying key GPU optimization techniques.

### Environment Setup
Verify your ROCm/HIP environment:

In [ ]:
!hipcc --version

In [ ]:
!rocminfo | grep -E 'Name|gfx'

---
## Problem: Matrix Multiplication C = A x B

For 4096x4096 FP32 matrices:
- **FLOPs**: 2 x 4096^3 = 137 trillion operations
- **Memory bandwidth needed** (naive): ~400 GB/s

## RDNA3 Memory Hierarchy
```
Global Memory (GMEM)    960 GB/s, high latency
         | load/store
LDS (Shared Mem)  ~29 TB/s, workgroup-shared (64KB per CU)
         | spill/fill
Registers     Per-thread, very fast (256 max per kernel)
```

---
## Baseline Performance Results (RX 7900 XTX)
| Kernel | Time | GFLOPS | vs rocBLAS |
|--------|------|------|----------|
| 0 - rocBLAS | 4.5ms | 30,547 | 100% |
| 1 - Naive | 136ms | 1,010 | 3.3% |
| 2 - LDS tiling | 34ms | 4,018 | 13.1% |
| 3 - Register tile | 6ms | 22,777 | 74.6% |
| 4 - GMEM dbl buffer | 5.4ms | 25,560 | 83.7% |
| 5 - LDS optimization | 4.1ms | 33,527 | **109.8%** |
| 6 - Dual FMA (ISA) | 3.6ms | 37,791 | **123.7%** |
| 7 - Loop unroll | 3.3ms | **41,256** | 135.1% |
| 8 - Batched GMEM | 2.8ms | **49,047** | **160.6%** |

---
## Kernel 1: Naive Implementation
**Approach**: One thread per C[i,j] element.

```cpp
__global__ void kernel1_naive(const float* A, const float* B, float* C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float acc = 0.0f;
        for (int k = 0; k < N; ++k)
            acc += A[row * K + k] * B[k * N + col];
        C[row * N + col] = acc;
    }
}
```

**Analysis**:
- Low arithmetic intensity: 2N FLOPs / (3 x N x 4 bytes) = 0.167 F/byte
- Non-coalesced memory access on B[]
- Same A[row] loaded redundantly by many threads

---
## Kernel 2: LDS Tiling
**Optimization**: Load tile into shared memory (LDS), reuse across wave.

```cpp
__shared__ float As[BK][BN], Bs[BK][BN];  // 32x32 tiles = 8KB
for (int t = 0; t < N; t += BK) {
    As[threadIdx.y][threadIdx.x] = A[N * row + t + threadIdx.x];
    Bs[threadIdx.y][threadIdx.x] = B[N * (threadIdx.y + t) + col];
    __syncthreads();  // Wait for full tile in LDS
    for (int k = 0; k < BK; ++k)
        sum += As[threadIdx.y][k] * Bs[k][threadIdx.x];
    __syncthreads();
}
```

**Benefits**:
- Reuses data: Each A[i,k] used 32 times (once per thread in row)
- Arithmetic intensity: ~5.3 F/byte = 32x improvement
- Coalesced loads: Neighboring threads load contiguous GMEM

---
## Kernel 3: Register Tiling
**Optimization**: Further tile in registers, compute 64 elements per thread (8x8).

Key changes from Kernel 2:
- Compute **64 elements per thread** (BLOCK_SIZE_M x BLOCK_SIZE_N = 8x8)
- Accumulate partial sums in registers: `float r[8][8]`
- Only write to GMEM once at the end

**Benefits**:
- Less LDS traffic (no intermediate reads/writes)
- Higher ILP: More independent ops for out-of-order execution
- Better occupancy: Wave stays busy while waiting on memory

---
## Kernel 4: GMEM Double Buffering
**Optimization**: Hide load latency with two LDS tiles in flight.

```cpp
// Pre-load next tile while computing current
As[(t/BK+1)%2][threadIdx.y][threadIdx.x] = A[...];  // Load tile T+1
__syncthreads();
sum += As[t/BK%2][...] * Bs[...] ;                   // Compute with tile t
```

**Benefits**:
- Latency hiding: GMEM takes ~400 cycles; computation overlaps
- VALU utilization higher

---
## Kernel 5: LDS Optimization
**Optimizations**:
1. **Padding**: `__shared__ float As[BK][BM+4]` - avoid bank conflicts
2. **CU mode**: `-mcumode` - use smaller workgroup size (64 threads)
3. **Larger wavefront tile**: 16x8 instead of 8x8

**Bank conflicts**:
LDS has 32 banks, each 4 bytes. Without padding threads access same bank.
Solution: `As[BK][BM + PADDING_DIV_4]` to stagger accesses.

**Result**: 33,527 GFLOPS - first kernel to beat rocBLAS!

---
## Kernel 6: ISA-level Dual FMA
**Optimization**: Directly write AMD GCN assembly for precise control.

```s
; Dual FMA executes 2 independent FMAs in one cycle
v_dual_fmac_f32 v10, v11, s[6:7], v12, v13 :: 
             v_dual_fmac_f32 v15, v16, s[6:7], v17, v18
```

**Dual-issue constraints**:
- Different VGPR banks (bank = reg_id % 4)
- Source registers from different banks execute in parallel

**Result**: 37,791 GFLOPS (+13% vs Kernel 5)

---
## How to Run the Full Test Suite

### Step 1: Build all kernels
```bash
cd ../fp32_sgemm_amd
./build.sh
```

In [ ]:
!cd ../fp32_sgemm_amd && ./build.sh


### Step 2: Run benchmark
```bash
cd ../fp32_sgemm_amd
./sgemm
```

---
## Profiling with rocprof
See what limits performance:

```bash
rocprofv3 --stats-db ./perf_stats.db ./sgemm
rocprof-parser stats.db | head -100
```

**Key metrics to watch**:
- GMEM throughput: Should approach 960 GB/s
- LDS utilization: Higher is better (80%+ ideal)
- VALU FMA32 Utilization: Target 75%+
- Wavefront stall reasons: Identify bottlenecks

---
## Summary: Optimization Journey

| Step | Technique | Key Insight |
|-----|---------|-------------|
| 1->2 | **LDS tiling** | Shared memory = fast cache for workgroup |
| 2->3 | Register tiling | Process multiple outputs per thread |
| 3->4 | **GMEM double buffer** | Overlap load and compute |
| 4->5 | **LDS bank fix + CU mode** | Avoid LDS stalls, better occupancy |
| 5->6 | **ISA dual FMA** | Precise register allocation |
| 6->7 | Loop unrolling | Schedule ops optimally |
| 7->8 | Batched loads | Reduce instruction overhead |

## References
- [Original Blog](https://seb-v.github.io/optimization/update/2025/01/20/Fast-GPU-Matrix-multiplication.html)
- [ROCm Documentation](https://rocm.docs.amd.com/)
- [RDNA3 ISA Guide](https://www.amd.com/en/developer/resources/rdna-architectures)